### <span style=color:blue> Joining listings_with_reviews and listings_with_calendar_dates    </span>

In [1]:
import sys
import json
import csv
import yaml

import importlib

import math

import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

from datetime import time
from datetime import date
from datetime import datetime
# with the above choices, the imported datetime.time(2023,07,01) is recognized
# from datetime import date
# from datetime import datetime

import pprint

import psycopg2
from sqlalchemy import create_engine, text as sql_text

# Create an utilities file util.py in a folder benchmarking and import it
# NOTE: I moved my util.py to the directory "helper_functions" -- seems like a better name
sys.path.append('../ECS116-HELPER-FUNCTIONS/')
import util

In [2]:
# test that utils.py has been imported well
util.hello_world()

'hello world'

<span style=color:blue>Getting mongodb connection set up</span>

In [3]:
from pymongo import MongoClient

client = MongoClient()
# could have written client = MongoClient("localhost", 27017)
#                 or client = MongoClient("mongodb://localhost:27017/")

# I already have a database "airbnb"
db = client.airbnb

# checking collections in airbnb 
print(db.list_collection_names())

['listings_small', 'listings_test', 'listings_with_calendar', 'listings_with_reviews', 'calendar', 'listings_with_reviews_m_3', 'listings_with_reviews_duplicate', 'reviews_3', 'listings_3']


In [5]:
db.calendar_by_ag.drop()
print(db.list_collection_names())

['listings_with_reviews_and_cal', 'calendar', 'listings_with_calendar', 'testing', 'listings_previously_built', 'reviews_3', 'listings_3', 'listings_with_reviews', 'calendar_previously_built', 'listings', 'listings_with_reviews_m_3']


In [4]:
print(f'Size of listings_with_reviews_m_3 is {db.listings_with_reviews_m_3.count_documents({})}.')
print(f'Size of listings_with_calendar is {db.listings_with_calendar.count_documents({})}.')

Size of listings_with_reviews_m_3 is 37434.
Size of listings_with_calendar is 37431.


In [5]:
pprint.pp(db.listings_with_reviews_m_3.find_one())

{'_id': ObjectId('682e75e57297e60d4a175744'),
 'id': '36121',
 'name': 'Lg Rm in Historic Prospect Heights',
 'host_id': '62165',
 'host_name': 'Michael',
 'neighbourhood_cleansed': 'Prospect Heights',
 'neighbourhood_group_cleansed': 'Brooklyn',
 'latitude': 40.67376,
 'longitude': -73.96611,
 'room_type': 'Private room',
 'price': 200.0,
 'minimum_nights': 90,
 'has_availability': 't',
 'number_of_reviews': 9,
 'number_of_reviews_ltm': 0,
 'last_review': datetime.datetime(2013, 5, 10, 0, 0),
 'license': '',
 'calculated_host_listings_count': 1,
 'reviews_per_month': 0.05,
 'reviews': [{'_id': ObjectId('682e75fc7297e60d4a185b98'),
              'listing_id': '36121',
              'id': '152196',
              'date': datetime.datetime(2010, 12, 11, 0, 0),
              'reviewer_id': '240039',
              'reviewer_name': 'Nikki',
              'comments': 'Michael is the man! Funny, smart and brilliant. His '
                          'place is awesome, in a cool neighborhood and 

In [6]:
pprint.pp(db.listings_with_calendar.find_one())

{'_id': '10000070',
 'average_price': 85.0,
 'first_available_date': datetime.datetime(2025, 3, 3, 0, 0),
 'last_available_date': datetime.datetime(2026, 3, 2, 0, 0),
 'dates_list': [{'date': datetime.datetime(2025, 3, 3, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 4, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 5, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30},
                {'date': datetime.datetime(2025, 3, 6, 0, 0),
                 'available': True,
                 'price': 85.0,
                 'minimum_nights': 30,
                 'maximum_nights': 30

In [7]:
db.listings_test1.drop()

pipeline = [
    { '$lookup': {
          'from': 'listings_with_calendar',
          'localField' : 'id',
          'foreignField' : '_id',
          'as' : 'cal_docs' 
        }
    },
    { '$out' : 'listings_test1' }
]

"""
result = db.listings_with_reviews.aggregate(pipeline)

pprint.pp(result.next())
"""

time1 = datetime.now()
result = db.listings_with_reviews.aggregate(pipeline)
time2 = datetime.now()
print(f'The time taken for this operation was {util.time_diff(time1,time2)} seconds.')

print()
print(db.list_collection_names())

size = db.listings_test1.count_documents({})
print(f'\nThe number of documents in listings_test1 is {size}.')

print()
pprint.pp(db.listings_test1.find_one())

The time taken for this operation was 40.434621 seconds.

['listings_with_calendar', 'testing', 'listings_previously_built', 'listings_test', 'listings_test1', 'listings_with_reviews', 'calendar_by_agg', 'calendar_previously_built', 'listings_with_reviews_and_cal', 'listings']

The number of documents in listings_test1 is 39202.

{'_id': ObjectId('6651189238b2bd10b4774432'),
 'id': '51944693',
 'name': 'Home in Queens · ★4.82 · 1 bedroom · 5 beds · 1 bath',
 'host_id': '91646104',
 'host_name': 'Pao',
 'neighbourhood_group': 'Queens',
 'neighbourhood': 'Woodside',
 'latitude': 40.74395,
 'longitude': -73.90858,
 'room_type': 'Entire home/apt',
 'price': 294.0,
 'minimum_nights': 30,
 'number_of_reviews': 57,
 'last_review': datetime.datetime(2023, 9, 24, 0, 0),
 'reviews_per_month': 1.98,
 'calculated_host_listings_count': 4,
 'availability_365': 89,
 'number_of_reviews_ltm': 23,
 'license': '',
 'reviews': [{'listing_id': '51944693',
              'review_id': '883354811516703393',
  

In [8]:
pprint.pp(db.listings_test1.find_one({'id': '35384734'}))

{'_id': ObjectId('664be01238b2bd10b477193c'),
 'id': '35384734',
 'name': 'Rental unit in New York · ★4.82 · Studio · 1 bed · 1 bath',
 'host_id': '266380288',
 'host_name': 'Rachel',
 'neighbourhood_group': 'Manhattan',
 'neighbourhood': 'West Village',
 'latitude': 40.73924,
 'longitude': -74.00366,
 'room_type': 'Entire home/apt',
 'price': nan,
 'minimum_nights': 30,
 'number_of_reviews': 22,
 'last_review': datetime.datetime(2023, 10, 1, 0, 0),
 'reviews_per_month': 0.41,
 'calculated_host_listings_count': 5,
 'availability_365': 0,
 'number_of_reviews_ltm': 2,
 'license': '',
 'reviews': [{'listing_id': '35384734',
              'review_id': '522113820',
              'date': datetime.datetime(2019, 9, 1, 0, 0),
              'reviewer_id': '69002790',
              'reviewer_name': 'Sanjay',
              'comments': 'The location is indeed spot on and very easily '
                          'approachable right in the heart of Chelsea '
                          '.<br/>Its a coz

In [9]:
# In the $unwind, using the option << "preserveNullAndEmptyArrays": true >>
#    so that we don't drop the listing with id = '35384734', which has empty array for cal_docs
# In general, if you do an $unwind, it converts a left outer join into a left (full) join,
#    because it removes documents that came from the left side, but have no matching records
#    from the right side.  By including the << "preserveNullAndEmptyArrays": true >> option,
#    you preserve the left join aspect of the $lookup
# This is following https://stackoverflow.com/questions/36725519/how-to-solve-empty-array-with-unwind

db.listings_test2.drop()

pipeline = [
    { '$lookup': {
          'from': 'listings_with_calendar',
          'localField' : 'id',
          'foreignField' : '_id',
          'as' : 'cal_docs' 
        }
    },
    { '$unwind': { 'path': '$cal_docs',  
                   'preserveNullAndEmptyArrays' : True
                 }
    },
    { '$out' : 'listings_test2'}
]

"""
result = db.listings_with_reviews.aggregate(pipeline)

pprint.pp(result.next())
"""
time1 = datetime.now()
result = db.listings_with_reviews.aggregate(pipeline)
time2 = datetime.now()
print(f'The time taken for this operation was {util.time_diff(time1,time2)} seconds.')

print()
print(db.list_collection_names())

size = db.listings_test2.count_documents({})
print(f'\nThe number of documents in listings_test2 is {size}.')

print()
pprint.pp(db.listings_test2.find_one())

The time taken for this operation was 36.539288 seconds.

['listings_with_calendar', 'testing', 'listings_previously_built', 'listings_test', 'listings_test1', 'listings_test2', 'listings_with_reviews', 'calendar_by_agg', 'calendar_previously_built', 'listings_with_reviews_and_cal', 'listings']

The number of documents in listings_test2 is 39202.

{'_id': ObjectId('6651189238b2bd10b4774432'),
 'id': '51944693',
 'name': 'Home in Queens · ★4.82 · 1 bedroom · 5 beds · 1 bath',
 'host_id': '91646104',
 'host_name': 'Pao',
 'neighbourhood_group': 'Queens',
 'neighbourhood': 'Woodside',
 'latitude': 40.74395,
 'longitude': -73.90858,
 'room_type': 'Entire home/apt',
 'price': 294.0,
 'minimum_nights': 30,
 'number_of_reviews': 57,
 'last_review': datetime.datetime(2023, 9, 24, 0, 0),
 'reviews_per_month': 1.98,
 'calculated_host_listings_count': 4,
 'availability_365': 89,
 'number_of_reviews_ltm': 23,
 'license': '',
 'reviews': [{'listing_id': '51944693',
              'review_id': '88335

In [10]:
pprint.pp(db.listings_test2.find_one({'id': '35384734'}))

{'_id': ObjectId('664be01238b2bd10b477193c'),
 'id': '35384734',
 'name': 'Rental unit in New York · ★4.82 · Studio · 1 bed · 1 bath',
 'host_id': '266380288',
 'host_name': 'Rachel',
 'neighbourhood_group': 'Manhattan',
 'neighbourhood': 'West Village',
 'latitude': 40.73924,
 'longitude': -74.00366,
 'room_type': 'Entire home/apt',
 'price': nan,
 'minimum_nights': 30,
 'number_of_reviews': 22,
 'last_review': datetime.datetime(2023, 10, 1, 0, 0),
 'reviews_per_month': 0.41,
 'calculated_host_listings_count': 5,
 'availability_365': 0,
 'number_of_reviews_ltm': 2,
 'license': '',
 'reviews': [{'listing_id': '35384734',
              'review_id': '522113820',
              'date': datetime.datetime(2019, 9, 1, 0, 0),
              'reviewer_id': '69002790',
              'reviewer_name': 'Sanjay',
              'comments': 'The location is indeed spot on and very easily '
                          'approachable right in the heart of Chelsea '
                          '.<br/>Its a coz

In [11]:
db.listings_test1.drop()

pipeline = [
    { '$lookup': {
          'from': 'listings_with_calendar',
          'localField' : 'id',
          'foreignField' : '_id',
          'as' : 'cal_docs' 
        }
    },
    { '$unwind': { 'path': '$cal_docs',  
                   'preserveNullAndEmptyArrays' : True
                 }
    },
    { '$addFields': 
          {'average_price': '$$ROOT.cal_docs.average_price',
           'first_available_date': '$$ROOT.cal_docs.first_available_date',
           'last_available_date':  '$$ROOT.cal_docs.last_available_date',
           'dates_list': '$$ROOT.cal_docs.dates_list'
          }
    },
    { '$out' : 'listings_test1'}
]

"""
result = db.listings_with_reviews.aggregate(pipeline)

pprint.pp(result.next())
"""

time1 = datetime.now()
result = db.listings_with_reviews.aggregate(pipeline)
time2 = datetime.now()
print(f'The time taken for this operation was {util.time_diff(time1,time2)} seconds.')

print()
print(db.list_collection_names())

size = db.listings_test1.count_documents({})
print(f'\nThe number of documents in listings_test1 is {size}.')

print()
pprint.pp(db.listings_test1.find_one())



The time taken for this operation was 64.452248 seconds.

['listings_with_calendar', 'testing', 'listings_previously_built', 'listings_test', 'listings_test2', 'listings_with_reviews', 'listings_test1', 'calendar_by_agg', 'calendar_previously_built', 'listings_with_reviews_and_cal', 'listings']

The number of documents in listings_test1 is 39202.

{'_id': ObjectId('6651189238b2bd10b4774432'),
 'id': '51944693',
 'name': 'Home in Queens · ★4.82 · 1 bedroom · 5 beds · 1 bath',
 'host_id': '91646104',
 'host_name': 'Pao',
 'neighbourhood_group': 'Queens',
 'neighbourhood': 'Woodside',
 'latitude': 40.74395,
 'longitude': -73.90858,
 'room_type': 'Entire home/apt',
 'price': 294.0,
 'minimum_nights': 30,
 'number_of_reviews': 57,
 'last_review': datetime.datetime(2023, 9, 24, 0, 0),
 'reviews_per_month': 1.98,
 'calculated_host_listings_count': 4,
 'availability_365': 89,
 'number_of_reviews_ltm': 23,
 'license': '',
 'reviews': [{'listing_id': '51944693',
              'review_id': '88335

In [ ]:
db.listings_with_reviews_and_cal.drop()

pipeline = [
    { '$lookup': {
          'from': 'listings_with_calendar',
          'localField' : 'id',
          'foreignField' : '_id',
          'as' : 'cal_docs' 
        }
    },
    { '$unwind': { 'path': '$cal_docs',  
                   'preserveNullAndEmptyArrays' : True
                 }
    },
    { '$addFields': 
          {'average_price': '$$ROOT.cal_docs.average_price',
           'first_available_date': '$$ROOT.cal_docs.first_available_date',
           'last_available_date':  '$$ROOT.cal_docs.last_available_date',
           'dates_list': '$$ROOT.cal_docs.dates_list'
          }
    },
    { '$unset': 'cal_docs' },     
    { '$out' : 'listings_with_reviews_and_cal'}
]
"""


"""
"""
result = db.listings_with_reviews.aggregate(pipeline)

pprint.pp(result.next())
"""

time1 = datetime.now()
result = db.listings_with_reviews.aggregate(pipeline)
time2 = datetime.now()
print(f'The time taken for this operation was {util.time_diff(time1,time2)} seconds.')

print()
print(db.list_collection_names())

size = db.listings_with_reviews_and_cal.count_documents({})
print(f'\nThe number of documents in listings_with_reviews_and_cal is {size}.')

In [9]:
print()
pprint.pp(db.listings_with_reviews_and_cal.find_one())


{'_id': ObjectId('682bd037744bf432bd8e9f66'),
 'id': '993726764319807393',
 'name': 'Cozy Place in Bushwick/Ridgewood!',
 'host_id': '86979892',
 'host_name': 'Genaro',
 'neighbourhood_cleansed': 'Ridgewood',
 'neighbourhood_group_cleansed': 'Queens',
 'latitude': 40.70380396680642,
 'longitude': -73.91047743662554,
 'room_type': 'Entire home/apt',
 'price': '$300.00',
 'minimum_nights': 2,
 'number_of_reviews': 19,
 'last_review': datetime.datetime(2025, 2, 16, 0, 0),
 'reviews_per_month': 2.78,
 'calculated_host_listings_count': 1,
 'has_availability': 't',
 'number_of_reviews_ltm': 19,
 'license': 'OSE-STRREG-0002236',
 'reviews': [{'listing_id': '993726764319807393',
              'review_id': '1261644208027538432',
              'date': datetime.datetime(2024, 10, 6, 0, 0),
              'reviewer_id': '207605361',
              'reviewer_name': 'Jayson',
              'comments': 'Great New York! Surprisingly quiet at night and was '
                          'walking distance f

In [10]:
db.listings_with_reviews_and_cal.drop()

pipeline = [
    { '$lookup': {
          'from': 'listings_with_calendar',
          'localField' : 'id',
          'foreignField' : '_id',
          'as' : 'cal_docs' 
        }
    },
    { '$unwind': { 'path': '$cal_docs',  
                   'preserveNullAndEmptyArrays' : True
                 }
    },
    { '$addFields': 
          {'average_price': '$$ROOT.cal_docs.average_price',
           'first_available_date': '$$ROOT.cal_docs.first_available_date',
           'last_available_date':  '$$ROOT.cal_docs.last_available_date',
           'dates_list': '$$ROOT.cal_docs.dates_list'
          }
    },
    { '$unset': 'cal_docs' },     
    { '$out' : 'listings_with_reviews_and_cal'}
]

time1 = datetime.now()
result = db.listings_with_reviews_m_3.aggregate(pipeline)
time2 = datetime.now()
print(f'The time taken for this operation was {util.time_diff(time1,time2)} seconds.')
# about 50 seconds, plus or minus

print()
print(db.list_collection_names())

size = db.listings_with_reviews_and_cal.count_documents({})
print(f'\nThe number of documents in listings_with_reviews_and_cal is {size}.')

print()
pprint.pp(db.listings_with_reviews_and_cal.find_one())


The time taken for this operation was 50.519691 seconds.

['calendar', 'listings_with_calendar', 'testing', 'listings_previously_built', 'reviews_3', 'listings_3', 'listings_with_reviews', 'calendar_previously_built', 'listings', 'listings_with_reviews_m_3', 'listings_with_reviews_and_cal']

The number of documents in listings_with_reviews_and_cal is 39202.

{'_id': ObjectId('665e91ad81a877ddad1a7549'),
 'id': '977395984065981849',
 'name': 'Home in Brooklyn · 1 bedroom · 1 bed · 1 bath',
 'host_id': '95344065',
 'host_name': 'Derek',
 'neighbourhood_group': 'Brooklyn',
 'neighbourhood': 'Sheepshead Bay',
 'latitude': 40.59179,
 'longitude': -73.94285,
 'room_type': 'Private room',
 'price': 30.0,
 'minimum_nights': 31,
 'number_of_reviews': 1,
 'last_review': datetime.datetime(2024, 1, 3, 0, 0),
 'reviews_per_month': 0.86,
 'calculated_host_listings_count': 7,
 'availability_365': 339,
 'number_of_reviews_ltm': 1,
 'license': '',
 'reviews': [{'_id': ObjectId('665e7a1981a877ddad19ca7b

In [12]:
# the one listing that is not in listings_with_calendar
pprint.pp(db.listings_with_reviews_and_cal.find_one({'id': '1105738108912824645'}))

# 38956942 - no reviews
# 1105738108912824645 - 1 review
# 38956986 - no reviews

{'_id': ObjectId('682bd037744bf432bd8ea457'),
 'id': '1105738108912824645',
 'name': 'Curated minibar & SMEG fridge',
 'host_id': '564300557',
 'host_name': 'Ace Hotel Brooklyn',
 'neighbourhood_cleansed': 'Boerum Hill',
 'neighbourhood_group_cleansed': 'Brooklyn',
 'latitude': 40.687819,
 'longitude': -73.983733,
 'room_type': 'Private room',
 'price': None,
 'minimum_nights': 1,
 'number_of_reviews': 1,
 'last_review': datetime.datetime(2024, 4, 29, 0, 0),
 'reviews_per_month': 0.1,
 'calculated_host_listings_count': 1,
 'has_availability': None,
 'number_of_reviews_ltm': 1,
 'license': 'Exempt',
 'reviews': [{'listing_id': '1105738108912824645',
              'review_id': '1145733536583551449',
              'date': datetime.datetime(2024, 4, 29, 0, 0),
              'reviewer_id': '162814052',
              'reviewer_name': 'Zean',
              'comments': 'Great place to stay if you’re spending most of your '
                          'time in both Brooklyn and Manhattan. The sta

In [13]:
result = db.listings_with_reviews_and_cal.find({'id': { '$regex': '^3538473.*$'}})
print(len(list(result)))

0


In [14]:
result = db.listings_with_reviews_and_cal.find({'id': { '$regex': '^35384...$'}})
print(len(list(result)))

1


In [15]:
result = db.listings_with_reviews_and_cal.find({'id': { '$regex': '^35384...$'}})
for doc in result:
    pprint.pp(doc)

{'_id': ObjectId('682bd037744bf432bd8edacb'),
 'id': '35384123',
 'name': 'Botanical Home',
 'host_id': '266360944',
 'host_name': 'Olga',
 'neighbourhood_cleansed': 'Springfield Gardens',
 'neighbourhood_group_cleansed': 'Queens',
 'latitude': 40.6617,
 'longitude': -73.76261,
 'room_type': 'Private room',
 'price': '$70.00',
 'minimum_nights': 2,
 'number_of_reviews': 124,
 'last_review': datetime.datetime(2025, 1, 8, 0, 0),
 'reviews_per_month': 1.78,
 'calculated_host_listings_count': 2,
 'has_availability': 't',
 'number_of_reviews_ltm': 29,
 'license': 'OSE-STRREG-0001756',
 'reviews': [{'listing_id': '35384123',
              'review_id': '470846731',
              'date': datetime.datetime(2019, 6, 16, 0, 0),
              'reviewer_id': '12952513',
              'reviewer_name': 'Wendy',
              'comments': 'I was the first guest at Olga and her mom’s Airbnb. '
                          'A very sweet encounter! All accommodations are '
                          'clean an

In [15]:
print(db.list_collection_names())

['calendar', 'listings_with_calendar', 'testing', 'listings_previously_built', 'reviews_3', 'listings_3', 'listings_with_reviews', 'calendar_previously_built', 'listings', 'listings_with_reviews_m_3', 'listings_with_reviews_and_cal']


In [16]:
db.listings_test1.drop()
db.listings_test2.drop()
print(db.listings_test.count_documents({}))
print(db.list_collection_names())

0
['calendar', 'listings_with_calendar', 'testing', 'listings_previously_built', 'reviews_3', 'listings_3', 'listings_with_reviews', 'calendar_previously_built', 'listings', 'listings_with_reviews_m_3', 'listings_with_reviews_and_cal']


### <span style=color:blue>Producing json output    </span>

In [19]:
size = db.listings_with_reviews_and_cal.count_documents({})
print(f'\nThe number of documents in listings_with_reviews_and_cal is {size}.')

print()
doc = db.listings_with_reviews_and_cal.find_one({})
pprint.pp(doc)


The number of documents in listings_with_reviews_and_cal is 37434.

{'_id': ObjectId('682bd037744bf432bd8e9f66'),
 'id': '993726764319807393',
 'name': 'Cozy Place in Bushwick/Ridgewood!',
 'host_id': '86979892',
 'host_name': 'Genaro',
 'neighbourhood_cleansed': 'Ridgewood',
 'neighbourhood_group_cleansed': 'Queens',
 'latitude': 40.70380396680642,
 'longitude': -73.91047743662554,
 'room_type': 'Entire home/apt',
 'price': '$300.00',
 'minimum_nights': 2,
 'number_of_reviews': 19,
 'last_review': datetime.datetime(2025, 2, 16, 0, 0),
 'reviews_per_month': 2.78,
 'calculated_host_listings_count': 1,
 'has_availability': 't',
 'number_of_reviews_ltm': 19,
 'license': 'OSE-STRREG-0002236',
 'reviews': [{'listing_id': '993726764319807393',
              'review_id': '1261644208027538432',
              'date': datetime.datetime(2024, 10, 6, 0, 0),
              'reviewer_id': '207605361',
              'reviewer_name': 'Jayson',
              'comments': 'Great New York! Surprisingly qu

In [20]:
# this function converts MongoDB docs in listings_with_reviews_and_cal into json storable
def convert_lwrc_to_json(doc):
    doc_new = {}
    # start by transferring all scalar keys over, then fix some of them
    for key in doc.keys():
        if key not in ['reviews', 'dates_list']:
            doc_new[key] = doc[key]
    # now fixing some possible issues
    doc_new['_id'] = str(doc['_id'])
    if doc['last_review'] == None:    # is null
        doc_new['last_review'] = None
    else:
        doc_new['last_review'] = doc['last_review'].strftime('%Y-%m-%d')
    if doc['price'] == None:
        doc_new['price'] = None
    # elif math.isnan(doc['price']):
    #     doc_new['price'] = None
    else:
        doc_new['price'] = doc['price']
    if math.isnan(doc['reviews_per_month']):
        doc_new['reviews_per_month'] = None
    else:
        doc_new['reviews_per_month'] = doc['reviews_per_month']
    # there is one document in the merger that has no calendar entries
    if 'first_available_date' in doc:
        if doc['first_available_date'] == None:
            doc_new['first_available_date'] = None
        else:
            doc_new['first_available_date'] = doc['first_available_date'].strftime('%Y-%m-%d')
    if 'last_available_date' in doc:
        if doc['last_available_date'] == None:
            doc_new['last_available_date'] = None
        else:
            doc_new['last_available_date'] = doc['last_available_date'].strftime('%Y-%m-%d')
    if 'average_price' in doc:
        doc_new['average_price'] = doc['average_price']
    
    # now dealing with the 'reviews' array
    rlist = []
    for r in doc['reviews']:
        r_new = {}
        # r_new['_id'] = str(r['_id'])
        r_new['date'] = r['date'].strftime('%Y-%m-%d')
        for key in r.keys():
            # I am cheating, and will rename the 'listing_id' column back to 'id'
            if key not in ['date', '_id']:
                r_new[key] = r[key]
        rlist.append(r_new)
    doc_new['reviews'] = rlist

    # now dealing with the 'dates_list' array
    dlist = []
    for d in doc['dates_list']:
        d_new = {}
        d_new['date'] = d['date'].strftime('%Y-%m-%d')
        for key in ['price', 'minimum_nights', 'maximum_nights', 'available']:
            d_new[key] = d[key]
        dlist.append(d_new)
    doc_new['dates_list'] = dlist
    
    return doc_new

# pprint.pp(doc)

pprint.pp(convert_lwrc_to_json(doc))

{'_id': '682bd037744bf432bd8e9f66',
 'id': '993726764319807393',
 'name': 'Cozy Place in Bushwick/Ridgewood!',
 'host_id': '86979892',
 'host_name': 'Genaro',
 'neighbourhood_cleansed': 'Ridgewood',
 'neighbourhood_group_cleansed': 'Queens',
 'latitude': 40.70380396680642,
 'longitude': -73.91047743662554,
 'room_type': 'Entire home/apt',
 'price': '$300.00',
 'minimum_nights': 2,
 'number_of_reviews': 19,
 'last_review': '2025-02-16',
 'reviews_per_month': 2.78,
 'calculated_host_listings_count': 1,
 'has_availability': 't',
 'number_of_reviews_ltm': 19,
 'license': 'OSE-STRREG-0002236',
 'average_price': 300.0,
 'first_available_date': '2025-03-02',
 'last_available_date': '2026-03-01',
 'reviews': [{'date': '2024-10-06',
              'listing_id': '993726764319807393',
              'review_id': '1261644208027538432',
              'reviewer_id': '207605361',
              'reviewer_name': 'Jayson',
              'comments': 'Great New York! Surprisingly quiet at night and was '
  

In [21]:
print(db.listings_with_reviews_and_cal.count_documents({}))

cursor = db.listings_with_reviews_and_cal.find({'id' : {'$regex' : '^1001.*$'}})
    
l = list(cursor)
print(len(l))

37434
28


In [22]:
cursor = db.listings_with_reviews_and_cal.find({'id' : {'$regex' : '^1001.*$'}})

output = []

for doc in cursor:
    output.append(convert_lwrc_to_json(doc))

print(len(output))

28


In [24]:
def write_dict_to_dir_json(dict, dir, filename):
    with open(dir  + filename, 'w') as fp:
        json.dump(dict, fp)


dir = '/Users/rick/DM-for-DS-2025/PA3-OUTPUT/'
filename = 'listings_with_reviews_and_cal_subset_1001.json'
write_dict_to_dir_json(output, dir, filename)

In [29]:
cursor = db.listings_with_reviews_and_cal.find({
    'average_price' : {
        '$gte' : 18370
    }
})

output = []

for doc in cursor:
    output.append(convert_lwrc_to_json(doc))

print(len(output))

157


In [31]:
dir = '/Users/rick/DM-for-DS-2025/PA3-OUTPUT/'
filename = 'listings_with_reviews_and_cal_subset_avg_price_18370.json'
write_dict_to_dir_json(output, dir, filename)